# Week 7 — Hands-On: Monte Carlo, MCMC, MCTS

이번 실습에서는 세 가지를 직접 구현합니다:
1) Monte Carlo로 π 추정
2) Metropolis-Hastings로 1D 분포 샘플링
3) MCTS로 간단한 게임(돌 가져가기) 의사결정


In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)
EPS = 1e-12  # numerical stability for denominator terms


## 1) Monte Carlo로 π 추정

In [ ]:
def estimate_pi(n_samples: int):
    xs = np.random.uniform(-1, 1, size=n_samples)
    ys = np.random.uniform(-1, 1, size=n_samples)
    inside = (xs**2 + ys**2) <= 1
    pi_hat = 4 * inside.mean()
    return pi_hat, xs, ys, inside

sample_sizes = [200, 1000, 5000, 20000]
records = []
for n in sample_sizes:
    pi_hat, _, _, _ = estimate_pi(n)
    records.append((n, pi_hat, abs(math.pi - pi_hat)))

for n, pi_hat, err in records:
    print(f\"N={n:6d} | pi_hat={pi_hat:.6f} | abs_error={err:.6f}\")


In [ ]:
N_VIS = 1500
pi_hat, xs, ys, inside = estimate_pi(N_VIS)

plt.figure(figsize=(6, 6))
plt.scatter(xs[inside], ys[inside], s=8, alpha=0.5, label='inside circle')
plt.scatter(xs[~inside], ys[~inside], s=8, alpha=0.5, label='outside circle')
circle = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=2)
plt.gca().add_patch(circle)
plt.xlim(-1.05, 1.05)
plt.ylim(-1.05, 1.05)
plt.gca().set_aspect('equal', adjustable='box')
plt.title(f'Monte Carlo pi estimate (N={N_VIS}, pi_hat={pi_hat:.4f})')
plt.legend()
plt.show()


## 2) MCMC (Metropolis-Hastings)로 분포 샘플링

In [ ]:
def target_unnormalized(x):
    return 0.35 * np.exp(-0.5 * ((x + 2.0) / 0.8) ** 2) + 0.65 * np.exp(-0.5 * ((x - 1.5) / 0.6) ** 2)

def metropolis(num_steps=15000, step_std=0.8, x0=0.0):
    x = x0
    current_density = target_unnormalized(x)  # cache current density to avoid recomputation
    samples = []
    accepts = 0

    for _ in range(num_steps):
        proposal = np.random.normal(x, step_std)
        proposal_density = target_unnormalized(proposal)
        alpha = min(1.0, proposal_density / (current_density + EPS))
        if np.random.rand() < alpha:
            x = proposal
            current_density = proposal_density
            accepts += 1
        samples.append(x)

    return np.array(samples), accepts / num_steps

samples, acc_rate = metropolis(num_steps=20000, step_std=0.9)
burn_in = 3000
post = samples[burn_in:]
print(f'Acceptance rate: {acc_rate:.3f}')
print(f'Posterior-like sample mean after burn-in: {post.mean():.3f}')


In [ ]:
xs = np.linspace(-6, 6, 500)
target = target_unnormalized(xs)
target = target / np.trapz(target, xs)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(samples[:300], lw=1)
plt.title('Trace plot (first 300 steps)')
plt.xlabel('step')

plt.subplot(1, 2, 2)
plt.hist(post, bins=50, density=True, alpha=0.65, label='MCMC samples')
plt.plot(xs, target, 'r-', lw=2, label='target density (normalized)')
plt.title('Histogram vs target distribution')
plt.legend()
plt.tight_layout()
plt.show()


## 3) MCTS로 간단한 게임 의사결정

In [ ]:
class Node:
    def __init__(self, stones, player_to_move, parent=None, action=None):
        self.stones = stones
        self.player_to_move = player_to_move
        self.parent = parent
        self.action = action
        self.children = []
        self.untried_actions = self.legal_actions()
        self.visits = 0
        self.value = 0.0

    def legal_actions(self):
        return [a for a in (1, 2) if a <= self.stones]

    def is_terminal(self):
        return self.stones == 0

    def expand(self):
        a = self.untried_actions.pop()
        child = Node(self.stones - a, -self.player_to_move, parent=self, action=a)
        self.children.append(child)
        return child

    def best_child_ucb(self, c=1.4):
        scores = []
        for ch in self.children:
            exploit = ch.value / (ch.visits + EPS)
            explore = c * math.sqrt(math.log(self.visits + 1) / (ch.visits + EPS))
            scores.append(exploit + explore)
        return self.children[int(np.argmax(scores))]

def rollout(stones, player_to_move):
    current_player = player_to_move
    s = stones
    while s > 0:
        a = random.choice([x for x in (1, 2) if x <= s])
        s -= a
        current_player *= -1
    winner = -current_player
    return winner

def mcts_search(n_stones=7, simulations=2000, c=1.4):
    root = Node(n_stones, player_to_move=1)

    for _ in range(simulations):
        node = root

        while not node.is_terminal() and len(node.untried_actions) == 0:
            node = node.best_child_ucb(c=c)

        if not node.is_terminal() and len(node.untried_actions) > 0:
            node = node.expand()

        winner = rollout(node.stones, node.player_to_move)

        while node is not None:
            node.visits += 1
            if winner == node.player_to_move:
                node.value += 1
            elif winner == -node.player_to_move:
                node.value -= 1
            node = node.parent

    stats = []
    for ch in root.children:
        stats.append({
            'action': ch.action,
            'visits': ch.visits,
            'mean_value': ch.value / (ch.visits + EPS)
        })
    stats = sorted(stats, key=lambda x: x['visits'], reverse=True)
    return stats

stats = mcts_search(n_stones=9, simulations=3000, c=1.2)
for s in stats:
    print(s)

best_action = stats[0]['action']
print(f'\nRecommended action at root: take {best_action} stone(s)')


## 4) Exercises

1. Monte Carlo π 추정에서 `sample_sizes`를 10배 키우고 오차 감소를 확인하세요.
2. MCMC에서 `step_std`를 0.2, 2.0으로 바꿔 acceptance rate과 trace plot을 비교하세요.
3. MCTS에서 `c` 값을 0.2, 2.0으로 바꿔 루트 방문 분포가 어떻게 달라지는지 확인하세요.
